# UABR-SLM · Notebook 06 — Benefit labels (θ0 = arm M), routing baselines, learned routers, gates G4–G6

**Phase 06** (PLAN §6, §10, §11; amendments A1, A5). GPU **T4** (×1 is enough) · ~60–80 min.

**θ0 = arm M** (`05_mixed/arm_M/model_final.pt`). *Documented deviation:* notebook 05's pre-set rule picked arm S
(subject-specific val EM 5.42 % vs 4.85 %, a noise-level margin), but S **lost accuracy on dose-256 facts**
(relation-adjusted −7.7 pp [−13.4, −1.6]); those errors would leak into half-B routing labels. M passes G1a on dosed
facts (+5.53 pp [+3.05, +8.10]) and shows no dose harm. Team decision, 10 Nov 2026.

**Steps**
1. **Benefit labels (A1)** with frozen θ0 for **half B** (router training), **validation** (λ, model selection, gates)
   and **test** (computed now with θ0 for the later retention evaluation; **sealed** — no test metric is printed here):
   greedy + k = 8 samples (T = 0.7, fixed seeds), closed-book and with evidence (non-overlapping top-3, overflow 3→2→1);
   `b_quality = mean F1(C) − mean F1(A)`; likelihood proxy p̄_C − p̄_A (secondary).
2. **λ** (team: keep b = 0 questions, label via b_net): label CALL iff `b_quality − λ > 0`. λ is chosen on validation
   from a 1/16 grid as the value whose labels best agree (balanced accuracy) with the realised greedy outcome
   "evidence strictly helps" (*my rule — the plan gives none*).
3. **Pre-retrieval routing baselines** (θ0): answer confidence (mean token log-prob, first-token prob), first-token
   entropy, hidden-state linear probe at `<ASSISTANT>`, popularity (s_pop), relation-only (reported, not required).
4. **Learned routers, both formats** (team): `action` (decision at `<ASSISTANT>`: `<ANSWER>` vs `<CALL>`) and
   `answer_slot` (`<CALL>` competes with the first answer token). Loss only at the decision position; trained on half B;
   best checkpoint by validation AUROC; **the format with the higher validation AUROC becomes θ_route,preSFT**.
   **v2 (team decision after run 1):** router training cut closed-book EM 12.4 % → 3.2 % (labels would go stale), so the
   tied embedding matrix is now **frozen** and every router step adds **half-A Q&A replay** (answer-token loss; data θ0 was
   already trained on). Only checkpoints keeping closed-book EM within **1 pp** of θ0 (300-question val subset) can be
   selected; the 2-logit loss is kept and declared (A5: the policy compares z_CALL vs z_ANSWER at `<ASSISTANT>`).
5. **Gates:** G4 (b tracks the greedy EM change), G5 (a routing signal exists), **G6b answer preservation** (closed-book
   and evidence EM of the routed model within 1 pp of θ0 on 500 val questions), **G6** (team rule: validation AUROC
   ≥ 0.65 continue · 0.58–0.65 continue with warning · < 0.58 stop). G1c/G1d are recomputed for arm M.

**Inputs:** outputs of notebooks **00, 01, 02, 03b, 05** (+ optionally a previous run of this notebook: its labels are reused). **Kaggle:** GPU T4 · Internet On · Save & Run All.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch, torch.nn.functional as F
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em
from uabr.eval.stats import clustered_bootstrap_mean
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.benefit import label_question, choose_lambda
from uabr.routing.metrics import auroc, auprc, utility_at_budget, oracle_utility_at_budget, clustered_bootstrap_auroc
from uabr.routing.router import FORMATS, router_prompt, router_scores, train_router, answer_slot_targets

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/06_routing" if ON_KAGGLE else os.path.abspath("outputs/06_routing")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"

K, TEMP, MAX_NEW, SEQ, SEED = 8, 0.7, 16, 512, 606
LAMBDA_GRID = np.round(np.arange(0, 0.5 + 1e-9, 1 / 16), 4)
BUDGETS = [0.05, 0.10, 0.20]
R_LR, R_EPOCHS, R_BS, R_EVAL = 3e-5, int(os.environ.get("R_EPOCHS", 3)), 32, int(os.environ.get("R_EVAL", 50))
G6_GO, G6_STOP = 0.65, 0.58
TOL_PP = 1.0                       # answer preservation tolerance (team decision)
REPLAY_BS, REPLAY_W = 16, 1.0      # half-A Q&A replay per router step
SEL_N, PRES_N = int(os.environ.get("SEL_N", 300)), int(os.environ.get("PRES_N", 500))
LIMIT = int(os.environ.get("LIMIT", 0))

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True)
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
facts = pd.read_parquet(find_input("03b_reinject/facts_doses.parquet", "FACTS"), columns=["id", "dose_group"])
THETA0 = find_input("05_mixed/arm_M/model_final.pt", "THETA0")
halfB_ids = set(int(x) for x in open(find_input("05_mixed/manifests/halfB_ids.txt", "HALFB")).read().split())
halfA_ids = set(int(x) for x in open(find_input("05_mixed/manifests/halfA_ids.txt", "HALFA")).read().split())
popqa = popqa.merge(facts, on="id", how="left"); popqa["dose_group"] = popqa["dose_group"].fillna("not_in_main")

WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
SETS = {"halfB": popqa[popqa.id.isin(halfB_ids)], "val": popqa[(popqa.split == "val") & popqa.in_main],
        "test": popqa[(popqa.split == "test") & popqa.in_main]}
SETS = {k: (v.head(LIMIT) if LIMIT else v).reset_index(drop=True) for k, v in SETS.items()}
assert not halfB_ids & halfA_ids
print(f"theta0 = {THETA0} | device {DEV} | " + " | ".join(f"{k} {len(v):,}" for k, v in SETS.items()))

## 3 · Benefit labels with frozen θ0 (half B, validation, test — test sealed)

In [ ]:
for name in SETS:                                   # reuse labels from an attached previous run of this notebook
    if not os.path.exists(f"{OUT}/labels_{name}.parquet"):
        prev = sorted(glob.glob(f"/kaggle/input/**/06_routing/labels_{name}.parquet", recursive=True))
        if prev:
            shutil.copy(prev[0], OUT); shutil.copy(os.path.join(os.path.dirname(prev[0]), f"hidden_{name}.npy"), OUT)
            print(f"reusing labels for {name} from {prev[0]}")
model = load_model(THETA0, DEV)
LAB = {}
for name, df in SETS.items():
    path = f"{OUT}/labels_{name}.parquet"
    if os.path.exists(path):
        LAB[name] = pd.read_parquet(path); stage(f"{name}: loaded {len(LAB[name]):,} cached labels"); continue
    rows = []
    for i, r in enumerate(df.itertuples(index=False)):
        sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
        rec = label_question(model, tok, r.question, list(r.gold_aliases), [WTEXT[d] for d in sel], r.id, SEED, K, TEMP,
                             MAX_NEW, SEQ, DEV, stop_ids, answer=r.answer)
        rec.update(id=r.id, evidence_doc_ids=sel[:rec["n_windows_used"]]); rows.append(rec)
        if (i + 1) % 1000 == 0: stage(f"  {name} {i+1:,}/{len(df):,}")
    lab = df[["id", "prop", "pop_bin", "log_pop", "dose_group", "group_id", "prior_correct", "split", "is_pilot", "question", "answer", "gold_aliases"]].merge(pd.DataFrame(rows), on="id")
    lab["greedy_first_id"] = lab["greedy_ids_A"].map(lambda g: int(g[0]) if len(g) else -1)
    np.save(f"{OUT}/hidden_{name}.npy", np.stack(lab["h_assistant"].values)); lab = lab.drop(columns=["h_assistant"])
    lab.to_parquet(path, index=False); LAB[name] = lab
    stage(f"{name}: {len(lab):,} labelled" + ("" if name == "test" else f" | closed EM {lab.em_A.mean():.1%} | evidence EM {lab.em_C.mean():.1%}"))
HID = {k: np.load(f"{OUT}/hidden_{k}.npy").astype(np.float32) for k in LAB}

## 4 · G1c / G1d recomputed for arm M (pilot ⊂ validation)

In [ ]:
pil = LAB["val"][LAB["val"].is_pilot]
G1C = float((pil.b_quality > 0).mean()); G1D = int(((pil.em_A == 1) & (pil.em_C == 0)).sum())
gain_pilot = clustered_bootstrap_mean((pil.em_A - pil.prior_correct) * 100, pil.group_id)
print(f"pilot (n={len(pil)}): closed EM {pil.em_A.mean():.1%} | evidence EM {pil.em_C.mean():.1%} | prior {pil.prior_correct.mean():.1%} "
      f"| gain {gain_pilot['mean']:+.1f} pp [{gain_pilot['lo']:+.1f}, {gain_pilot['hi']:+.1f}] | P(b>0) {G1C:.1%} | damage {G1D}")

## 5 · Choose λ on validation; labels; benefit distribution; G4

In [ ]:
V = LAB["val"]
LAM = choose_lambda(V.b_quality.values, V.delta_greedy_em.values, LAMBDA_GRID)
lam = LAM["chosen"]["lambda"]
print(f"chosen lambda = {lam} (balanced acc {LAM['chosen']['balanced_acc']:.3f}, val CALL-label rate {LAM['chosen']['call_rate']:.1%})")
display(pd.DataFrame(LAM["grid"]).round(3).T)
for k in LAB:
    LAB[k]["b_net"] = LAB[k].b_quality - lam
    LAB[k]["y"] = (LAB[k].b_net > 0).astype(int)
    LAB[k]["outcome"] = np.select([(LAB[k].em_A == 1) & (LAB[k].em_C == 1), LAB[k].em_A == 1, LAB[k].em_C == 1],
                                  ["both", "only_answer", "only_call"], "neither")
if os.environ.get("TEST_FAKE_Y") and not ON_KAGGLE:       # LOCAL TESTS ONLY: plant learnable labels (toy model answers nothing)
    for k in LAB:
        LAB[k]["y"] = ((LAB[k].prop.astype("category").cat.codes % 2) ^ (np.random.default_rng(1).random(len(LAB[k])) < 0.2)).astype(int)
    print("!!! TEST_FAKE_Y: labels replaced for a local smoke test - never on Kaggle")
assert LAB["halfB"].y.nunique() == 2 and LAB["val"].y.nunique() == 2, \
    "labels have a single class - routing cannot be trained/evaluated (check theta0 / lambda)"
for k in ["halfB", "val"]:
    L = LAB[k]; b = L.b_quality
    print(f"{k}: CALL labels {L.y.mean():.1%} | P(|b|<=0.01) {(b.abs()<=0.01).mean():.1%} | P(|b|<=0.05) {(b.abs()<=0.05).mean():.1%} | "
          f"P(|b|<=0.10) {(b.abs()<=0.10).mean():.1%} | outcomes {L.outcome.value_counts(normalize=True).round(3).to_dict()}")
display(V.groupby("prop").agg(n=("id", "size"), call_label=("y", "mean"), mean_b=("b_quality", "mean"), em_A=("em_A", "mean"), em_C=("em_C", "mean")).round(3))
display(V.groupby("pop_bin").agg(call_label=("y", "mean"), mean_b=("b_quality", "mean"), em_A=("em_A", "mean"), em_C=("em_C", "mean")).round(3))
from scipy.stats import spearmanr
rho = spearmanr(V.b_quality, V.delta_greedy_em).correlation if V.delta_greedy_em.nunique() > 1 else float("nan")
rng = np.random.default_rng(0); grp = V.group_id.values; ug = np.unique(grp); gi = {g: np.where(grp == g)[0] for g in ug}; bs = []
for _ in range(1000):
    idx = np.concatenate([gi[g] for g in rng.choice(ug, len(ug))]); bs.append(spearmanr(V.b_quality.values[idx], V.delta_greedy_em.values[idx]).correlation)
bs = [x for x in bs if np.isfinite(x)]
RHO = {"rho": float(rho), "lo": float(np.quantile(bs, .025)) if bs else float("nan"), "hi": float(np.quantile(bs, .975)) if bs else float("nan")}
rho_lik = spearmanr(V.b_quality, V.b_likelihood).correlation
print(f"G4: Spearman(b_quality, greedy EM change) = {rho:.3f} [{RHO['lo']:.3f}, {RHO['hi']:.3f}] | Spearman(b_quality, likelihood proxy) = {rho_lik:.3f}")

## 6 · Pre-retrieval routing baselines on validation (θ0)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
B, V = LAB["halfB"], LAB["val"]
yB, yV = B.y.values, V.y.values
sc = StandardScaler().fit(HID["halfB"])
probe = LogisticRegression(max_iter=2000, C=0.1).fit(sc.transform(HID["halfB"]), yB)
rel_rate = B.groupby("prop").y.mean()
BASE_SCORES = {
    "neg_answer_conf (mean lp)": -V.conf_mean_lp.fillna(V.conf_mean_lp.min()).values,
    "neg_first_token_prob": -V.conf_first_p.fillna(0).values,
    "first_token_entropy": V.first_entropy.values,
    "hidden_probe@<ASSISTANT>": probe.predict_proba(sc.transform(HID["val"]))[:, 1],
    "neg_popularity (s_pop)": -V.log_pop.values,
    "relation_only (half-B CALL rate)": V.prop.map(rel_rate).fillna(B.y.mean()).values + 1e-6 * np.random.default_rng(1).random(len(V)),
    "random": np.random.default_rng(2).random(len(V)),
}
RA, RC = V.em_A.values, V.em_C.values
def policy_row(name, s):
    a = clustered_bootstrap_auroc(yV, s, V.group_id.values, n_boot=1000)
    row = {"policy": name, "AUROC": a["auroc"], "lo": a["lo"], "hi": a["hi"], "AUPRC": auprc(yV, s)}
    for bud in BUDGETS:
        u = utility_at_budget(s, RA, RC, lam, bud); row[f"U@{int(bud*100)}%"] = u["utility"]; row[f"acc@{int(bud*100)}%"] = u["accuracy"]
    return row
rows = [policy_row(n, s) for n, s in BASE_SCORES.items()]
rows.append({"policy": "always answer", **{f"U@{int(b*100)}%": RA.mean() for b in BUDGETS}})
rows.append({"policy": "oracle (within budget)", **{f"U@{int(b*100)}%": oracle_utility_at_budget(RA, RC, lam, b) for b in BUDGETS}})
BASELINES = pd.DataFrame(rows); display(BASELINES.round(3))
G5_best = BASELINES[BASELINES.policy.isin(["neg_answer_conf (mean lp)", "neg_first_token_prob", "first_token_entropy", "hidden_probe@<ASSISTANT>"])].sort_values("AUROC").iloc[-1]
print(f"G5 best pre-retrieval signal: {G5_best.policy} AUROC {G5_best.AUROC:.3f} [{G5_best.lo:.3f}, {G5_best.hi:.3f}]")

## 7 · Learned routers: `action` and `answer_slot` (trained on half B, selected on validation)

In [ ]:
@torch.no_grad()
def greedy_ban(model, ids, banned=(T.CALL_ID,), max_new=MAX_NEW):
    x = torch.tensor([ids], device=DEV); out = []
    for _ in range(max_new):
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            z = model(x[:, -512:])[0][0, -1].float()
        z[list(banned)] = -1e9; t = int(z.argmax())
        if t == T.EOS_ID or "\n" in tok.decode([t]): break
        out.append(t); x = torch.cat([x, torch.tensor([[t]], device=DEV)], 1)
    return PR.clean_generation(tok.decode(out))

def closed_em(model, df):
    model.eval(); return float(np.mean([alias_em(greedy_ban(model, Q.closed_prompt_ids(tok, q)), g) for q, g in zip(df.question, df.gold_aliases)]))

def evidence_em(model, df):
    model.eval(); out = []
    for r in df.itertuples(index=False):
        sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
        p, _ = Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)
        out.append(alias_em(greedy_ban(model, p), r.gold_aliases))
    return float(np.mean(out))

# half-A replay examples (closed + evidence; theta0 was already trained on these)
Arows = popqa[popqa.id.isin(halfA_ids)]
if LIMIT: Arows = Arows.head(LIMIT)
REPLAY = []
for r in Arows.itertuples(index=False):
    REPLAY.append(Q.closed_example(tok, r.question, r.answer))
    sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
    REPLAY.append(Q.evidence_example(tok, r.question, [WTEXT[d] for d in sel], r.answer, SEQ)[0])
SEL = V.sample(min(SEL_N, len(V)), random_state=4); PRES = V.sample(min(PRES_N, len(V)), random_state=3)
m0 = load_model(THETA0, DEV)
EM0_SEL = closed_em(m0, SEL); EM0_C, EM0_E = closed_em(m0, PRES), evidence_em(m0, PRES); del m0
print(f"theta0: closed EM {EM0_SEL:.1%} (selection subset) | {EM0_C:.1%} closed / {EM0_E:.1%} evidence (preservation set) | replay examples {len(REPLAY):,}")

ROUTERS = {}
for fmt in FORMATS:
    m = load_model(THETA0, DEV)
    trP = [router_prompt(tok, q, fmt) for q in B.question]; vaP = [router_prompt(tok, q, fmt) for q in V.question]
    trT = list(yB) if fmt == "action" else answer_slot_targets(yB, B.greedy_first_id.values)
    def qualify(model, _f=fmt):
        e = closed_em(model, SEL); return e >= EM0_SEL - TOL_PP / 100, {"closed_em_sel": round(e, 4)}
    stage(f"training router [{fmt}] on {len(trP):,} half-B questions (CALL labels {yB.mean():.1%}) | frozen embeddings + half-A replay")
    best, hist = train_router(m, trP, trT, vaP, yV, fmt, DEV, lr=R_LR, epochs=R_EPOCHS, bs=R_BS, eval_every=R_EVAL, seed=SEED, log=stage,
                              freeze_embed=True, replay_examples=REPLAY, replay_bs=REPLAY_BS, replay_weight=REPLAY_W, qualify=qualify)
    m.load_state_dict(best["state"]); s, mg = router_scores(m, vaP, fmt, DEV)
    emc, eme = closed_em(m, PRES), evidence_em(m, PRES)
    keep = (emc >= EM0_C - TOL_PP / 100) and (eme >= EM0_E - TOL_PP / 100)
    torch.save({"model": best["state"], "model_cfg": m.cfg.to_dict(), "format": fmt, "best_step": best["step"]}, f"{OUT}/router_{fmt}.pt")
    ROUTERS[fmt] = {"score": s, "margin": mg, "best_step": best["step"], "hist": hist, "model": m, "em_closed": emc, "em_evidence": eme, "preserved": keep}
    print(f"[{fmt}] best val AUROC {auroc(yV, s):.4f} at step {best['step']} | closed EM {emc:.1%} (theta0 {EM0_C:.1%}) | "
          f"evidence EM {eme:.1%} (theta0 {EM0_E:.1%}) | preserved within {TOL_PP} pp: {keep}")
for fmt in FORMATS:
    rows.append(policy_row(f"learned router [{fmt}]", ROUTERS[fmt]["score"]))
POLICIES = pd.DataFrame(rows); display(POLICIES.round(3))

## 8 · Choose θ_route,preSFT (rule: higher validation AUROC) + answer quality after routing (A5)

In [ ]:
cands = [f for f in FORMATS if ROUTERS[f]["preserved"]] or list(FORMATS)
FMT = max(cands, key=lambda f: auroc(yV, ROUTERS[f]["score"]))
R = ROUTERS[FMT]; RM = R["model"]
thr = {f"{int(b*100)}%": float(np.quantile(R["score"], 1 - b)) for b in BUDGETS}
json.dump({"format": FMT, "lambda": lam, "thresholds_val": thr, "checkpoint": f"06_routing/router_{FMT}.pt", "theta0": THETA0,
           "decision": "compare z_CALL vs z_ANSWER at <ASSISTANT> (2-logit, constrained)" if FMT == "action" else "logit P(<CALL>) at the first answer position",
           "rule": "among formats preserving answer EM within 1 pp, the higher validation AUROC"}, open(f"{OUT}/routing_policy.json", "w"), indent=2)
print(f"theta_route,preSFT = [{FMT}] | preserved formats: {cands if all(ROUTERS[f]['preserved'] for f in cands) else 'NONE (flagged)'} | thresholds {thr}")
emit = []
for q in PRES.question.head(300):                                        # A5 (i)
    with torch.no_grad():
        z = RM(torch.tensor([Q.closed_prompt_ids(tok, q)[:-1]], device=DEV))[0][0, -1].float()
    emit.append(int(z.argmax()) in (T.ANSWER_ID, T.CALL_ID))
em_before, em_after = EM0_C, R["em_closed"]
print(f"A5: closed EM theta0 {EM0_C:.1%} -> routed {R['em_closed']:.1%} | evidence EM {EM0_E:.1%} -> {R['em_evidence']:.1%} | "
      f"free-generation action-token emission after <ASSISTANT>: {np.mean(emit):.1%} (2-logit loss; declared)")

## 9 · Policy analysis on validation (chosen router)

In [ ]:
s = R["score"]
V["router_score"] = s; V["margin"] = R["margin"]
V["call@10%"] = (s >= thr["10%"]).astype(int)
display(V.groupby("dose_group")[["call@10%", "y"]].mean().round(3).T)
display(V.groupby("pop_bin")[["call@10%", "y"]].mean().round(3).T)
per_rel = {r: auroc(g.y, g.router_score) for r, g in V.groupby("prop") if g.y.nunique() == 2 and len(g) >= 20}
print("per-relation AUROC:", {k: round(v, 3) for k, v in per_rel.items()})
print("macro per-relation AUROC:", round(float(np.nanmean(list(per_rel.values()))), 3))
print("margin quantiles (UABR m_i):", np.quantile(V.margin, [.1, .25, .5, .75, .9]).round(3))

## 10 · Save

In [ ]:
for k in LAB: LAB[k].to_parquet(f"{OUT}/labels_{k}.parquet", index=False)
V[["id", "y", "b_quality", "router_score", "margin", "call@10%"]].to_parquet(f"{OUT}/val_router_scores.parquet", index=False)
POLICIES.to_csv(f"{OUT}/policies_val.csv", index=False)
json.dump({f: ROUTERS[f]["hist"] for f in FORMATS}, open(f"{OUT}/router_histories.json", "w"), indent=2)
for f in FORMATS:
    if f != FMT and os.path.exists(f"{OUT}/router_{f}.pt"): os.rename(f"{OUT}/router_{f}.pt", f"{OUT}/router_{f}_not_chosen.pt")
json.dump({"notebook": "06_routing", "theta0": THETA0, "lambda": LAM, "G4": RHO, "spearman_b_vs_likelihood": rho_lik,
           "pilot": {"gain_pp": gain_pilot, "P_b_pos": G1C, "damage": G1D}, "chosen_format": FMT,
           "router_val_auroc": {f: auroc(yV, ROUTERS[f]["score"]) for f in FORMATS}, "per_relation_auroc": per_rel,
           "answer_preservation": {"theta0": [EM0_C, EM0_E], **{f: [ROUTERS[f]["em_closed"], ROUTERS[f]["em_evidence"], ROUTERS[f]["preserved"]] for f in FORMATS}},
           "router_best_steps": {f: ROUTERS[f]["best_step"] for f in FORMATS}, "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}},
          open(f"{OUT}/meta.json", "w"), indent=2, default=str)
for f in sorted(glob.glob(f"{OUT}/*")):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1e6:9.1f} MB  {os.path.basename(f)}")

## 11 · Gates G4, G5, G6 (+ G1c/G1d for arm M)

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("G1c_M_b_not_almost_all_positive", G1C < 0.90, f"P(b>0) = {G1C:.1%} (pilot, arm M)")
add("G1d_M_retrieval_damage_exists", G1D >= 1, f"{G1D} questions")
add("G4_b_tracks_greedy_EM_change", RHO["lo"] > 0, f"Spearman {RHO['rho']:.3f} [{RHO['lo']:.3f}, {RHO['hi']:.3f}]")
add("G5_routing_signal_exists", G5_best.lo > 0.5, f"{G5_best.policy}: AUROC {G5_best.AUROC:.3f} [{G5_best.lo:.3f}, {G5_best.hi:.3f}]")
add("G6b_answer_preservation_1pp", R["preserved"],
    f"closed {EM0_C:.1%} -> {R['em_closed']:.1%} | evidence {EM0_E:.1%} -> {R['em_evidence']:.1%} (tolerance {TOL_PP} pp)")
ra = clustered_bootstrap_auroc(yV, s, V.group_id.values, n_boot=1000)
g6 = "GO" if ra["auroc"] >= G6_GO else ("CONTINUE_WITH_WARNING" if ra["auroc"] >= G6_STOP else "STOP")
add("G6_router_val_AUROC", ra["auroc"] >= G6_STOP, f"[{FMT}] {ra['auroc']:.3f} [{ra['lo']:.3f}, {ra['hi']:.3f}] -> {g6}")
add("G6_go_threshold_0.65", ra["auroc"] >= G6_GO, f"{ra['auroc']:.3f} vs 0.65 (team: 0.58-0.65 continues)", "warn")
rel = POLICIES.set_index("policy").loc["relation_only (half-B CALL rate)", "AUROC"]
add("info_vs_relation_only", True, f"router {ra['auroc']:.3f} vs relation-only {rel:.3f} (reported, not required)", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "G4_G5_G6", "status": status, "G6": g6, "format": FMT, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print(f"\nGATES 06: {status} | G6 decision: {g6}")

## Next
- **G6 = GO / CONTINUE_WITH_WARNING:** notebook 07 = Phase 5 SFT-only pilot from θ_route,preSFT (`routing_policy.json`):
  E0 (frozen) vs E1 (plain SFT), flip rates by margin (A4), action-slot collapse checks (A5) → GO / RECALIBRATION_ONLY / NO_CHANGE.
- **G6 = STOP:** routing at 40M on PopQA is not learnable enough; report it and fall back to the synthetic study as the main result.